# V17 - Bloco 1A: publico com possivel elegibilidade

Esta versao e uma fase isolada da V17 e nao altera a V16. O notebook constroi somente a primeira feature do Bloco 1A: `CD_CLI` e `TS_ATL_TRAN_REF`.

A regra funcional aprovada e `CD_TIP_PSS = 1` e janela de `TS_ATL_TRAN`. O resultado ainda nao representa elegibilidade final. Nao ha leitura de blocos posteriores nem escrita externa.

In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_spark = GerenciadorLocal(
        nome_sessao="ana-edu-fin-cli-v17-bloco-1a",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": "MODELAGEM",
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    # Usa somente o padrao da infraestrutura; tuning sera tratado apos a medicao.
    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        jars=["/dados/shared/bin/ojdbc8.jar"],
    )
    print("[V17][BLOCO_1A] Sessao Spark inicializada pelo padrao corporativo.")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise

## Utilitarios corporativos

Reutiliza os gerenciadores existentes de sessao e conexao. Nenhuma infraestrutura nova e criada.

In [ ]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb

## Parametros minimos

Nenhuma janela ou parametro operacional e inventado. A leitura DB2 sera interrompida se qualquer valor obrigatorio estiver ausente.

In [ ]:
%%spark

import calendar
import datetime
import os
from datetime import timedelta

FONTE_BLOCO_1A = "DB2GFP.TRAN_RLZD_INST_PCT"
FILTROS_APROVADOS = True
DATA_EXECUCAO = datetime.date.fromisoformat(str(obter_variavel_ambiente("HOJE"))[:10])

def recuar_meses_sem_biblioteca(data_referencia, quantidade_meses):
    total_meses = data_referencia.year * 12 + (data_referencia.month - 1) - quantidade_meses
    ano, mes_zero_based = divmod(total_meses, 12)
    mes = mes_zero_based + 1
    ultimo_dia = calendar.monthrange(ano, mes)[1]
    dia = min(data_referencia.day, ultimo_dia)
    return datetime.date(ano, mes, dia)

DATA_INICIAL_PUBLICO = recuar_meses_sem_biblioteca(DATA_EXECUCAO, 1)
DATA_FINAL_EXCLUSIVA_PUBLICO = DATA_EXECUCAO + timedelta(days=1)
FETCHSIZE = os.environ.get("V17_FETCHSIZE")
QUERY_TIMEOUT_SECONDS = os.environ.get("V17_QUERY_TIMEOUT_SECONDS")

print(f"[V17][BLOCO_1A] DATA_EXECUCAO = {DATA_EXECUCAO}")
print(f"[V17][BLOCO_1A] DATA_INICIAL_PUBLICO = {DATA_INICIAL_PUBLICO or 'PENDENTE'}")
print(f"[V17][BLOCO_1A] DATA_FINAL_EXCLUSIVA_PUBLICO = {DATA_FINAL_EXCLUSIVA_PUBLICO or 'PENDENTE'}")
print(f"[V17][BLOCO_1A] FETCHSIZE = {FETCHSIZE or 'PENDENTE'}")
print(f"[V17][BLOCO_1A] QUERY_TIMEOUT_SECONDS = {QUERY_TIMEOUT_SECONDS or 'PENDENTE'}")
print("[V17][BLOCO_1A] configuracao operacional minima carregada.")

## Contrato do Bloco 1A

**Objetivo:** produzir o primeiro publico com possivel elegibilidade: clientes PF com registros incluidos no sistema dentro da janela aprovada.

**Entrada:** transacoes fisicas de `DB2GFP.TRAN_RLZD_INST_PCT`.

**Regra funcional unica:** `CD_TIP_PSS = 1` e `DATA_INICIAL_PUBLICO <= TS_ATL_TRAN < DATA_FINAL_EXCLUSIVA_PUBLICO`.

**Saida:** uma linha por `CD_CLI`, contendo `CD_CLI` e `TS_ATL_TRAN_REF = MAX(TS_ATL_TRAN)`.

**Destino:** views/DataFrames temporarios. Escrita externa: nao. `NR_PTC` participa somente da hipotese tecnica da Estrategia B.

Nao fazem parte desta fase: CPF, conta, produto, data economica, renda, DVS, ciclo, perfil, categoria ou target final. `TS_ATL_TRAN_REF` permanece com o timestamp original; nenhum `DT_REF` e criado.

## Historico de decisao temporal

A decisao anterior baseada em `TS_INCL_TRAN` foi descartada porque essa coluna nao consta na documentacao da fonte. A V17 usa `TS_ATL_TRAN`, que e a coluna temporal comprovada.

## Regra aprovada e gate do Bloco 1A

- `CD_TIP_PSS = 1`: aprovado para o Bloco 1A.
- Janela de `TS_ATL_TRAN`: aprovada para o Bloco 1A.
- `NR_PTC`: candidato exclusivamente tecnico da Estrategia B.
- Nenhum outro filtro funcional e aplicado.

In [ ]:
%%spark

def validar_gate_bloco_1a():
    if FILTROS_APROVADOS is not True:
        raise RuntimeError("BLOQUEADO: filtros do Bloco 1A ainda nao foram aprovados.")

def validar_parametros_bloco_1a():
    exigidos = {
        "DATA_INICIAL_PUBLICO": DATA_INICIAL_PUBLICO,
        "DATA_FINAL_EXCLUSIVA_PUBLICO": DATA_FINAL_EXCLUSIVA_PUBLICO,
        "FETCHSIZE": FETCHSIZE,
        "QUERY_TIMEOUT_SECONDS": QUERY_TIMEOUT_SECONDS,
    }
    ausentes = [nome for nome, valor in exigidos.items() if valor in (None, "")]
    if ausentes:
        raise RuntimeError("BLOQUEADO: parametros do Bloco 1A ausentes: " + ", ".join(ausentes))
    try:
        inicio = datetime.date.fromisoformat(str(DATA_INICIAL_PUBLICO)[:10])
        fim = datetime.date.fromisoformat(str(DATA_FINAL_EXCLUSIVA_PUBLICO)[:10])
        fetch = int(FETCHSIZE)
        timeout = int(QUERY_TIMEOUT_SECONDS)
    except (TypeError, ValueError) as exc:
        raise RuntimeError(f"BLOQUEADO: parametro invalido do Bloco 1A: {exc}") from exc
    if inicio >= fim or fetch <= 0 or timeout <= 0:
        raise RuntimeError("BLOQUEADO: limites dos parametros do Bloco 1A sao invalidos.")
    return inicio.isoformat(), fim.isoformat(), fetch, timeout

validar_gate_bloco_1a()
print("[V17][BLOCO_1A] Gate liberado somente para esta feature.")

## Estrategias A e B

A consulta A agrupa no DB2 por cliente. A consulta B repete os mesmos filtros em quatro faixas tecnicas de `NR_PTC`; depois as faixas sao unidas e o maior timestamp e calculado no Spark. O campo tecnico nao integra a feature.

In [ ]:
%%spark

import os
import time
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

data_ini_sql, data_fim_sql, fetch_efetivo, timeout_efetivo = validar_parametros_bloco_1a()
conector_db2 = criar_conector_db2_spark(env=dict(os.environ))
NR_PTC_RANGES = [(1, 25, "G1"), (26, 50, "G2"), (51, 75, "G3"), (76, 100, "G4")]

def validar_sql_bloco_1a(sql, com_faixa=False):
    sql_up = sql.upper()
    if "CD_TIP_PSS = 1" not in sql_up or "TIMESTAMP(TS_ATL_TRAN) >=" not in sql_up or "TIMESTAMP(TS_ATL_TRAN) <" not in sql_up:
        raise AssertionError("SQL sem a regra funcional aprovada.")
    if "NR_PTC" in sql_up and not com_faixa:
        raise AssertionError("NR_PTC so pode aparecer na Estrategia B.")

def validar_resultado(df, nome):
    r = df.agg(
        F.count(F.lit(1)).alias("QT_LINHAS"),
        F.countDistinct("CD_CLI").alias("QT_CD_CLI"),
        F.sum(F.when(F.col("CD_CLI").isNull(), 1).otherwise(0)).alias("QT_CD_CLI_NULOS"),
        F.sum(F.when(F.col("TS_ATL_TRAN_REF").isNull(), 1).otherwise(0)).alias("QT_TS_NULOS"),
        F.min("TS_ATL_TRAN_REF").alias("MENOR_TS_ATL_TRAN_REF"),
        F.max("TS_ATL_TRAN_REF").alias("MAIOR_TS_ATL_TRAN_REF"),
    ).first().asDict()
    linhas = int(r["QT_LINHAS"] or 0)
    clientes = int(r["QT_CD_CLI"] or 0)
    nulos_cli = int(r["QT_CD_CLI_NULOS"] or 0)
    nulos_ts = int(r["QT_TS_NULOS"] or 0)
    duplicados = linhas - clientes
    if nulos_cli or nulos_ts or duplicados:
        raise AssertionError(f"{nome}: validacao falhou: {r}, duplicados={duplicados}")
    return {"qt_linhas": linhas, "qt_cd_cli": clientes, "qt_cd_cli_nulos": nulos_cli, "qt_ts_nulos": nulos_ts, "qt_duplicados_cd_cli": duplicados, "menor_ts_atl_tran_ref": r["MENOR_TS_ATL_TRAN_REF"], "maior_ts_atl_tran_ref": r["MAIOR_TS_ATL_TRAN_REF"]}

sql_a = f"""
SELECT CD_CLI, MAX(TIMESTAMP(TS_ATL_TRAN)) AS TS_ATL_TRAN_REF
FROM {FONTE_BLOCO_1A}
WHERE CD_TIP_PSS = 1
  AND TIMESTAMP(TS_ATL_TRAN) >= TIMESTAMP('{data_ini_sql} 00:00:00')
  AND TIMESTAMP(TS_ATL_TRAN) < TIMESTAMP('{data_fim_sql} 00:00:00')
GROUP BY CD_CLI
"""
validar_sql_bloco_1a(sql_a)
inicio_a = time.perf_counter()
df_a = conector_db2.sql(sql_a, fetchsize=fetch_efetivo, query_timeout=timeout_efetivo).select("CD_CLI", "TS_ATL_TRAN_REF").persist(StorageLevel.MEMORY_AND_DISK)
linhas_a_jdbc = df_a.count()
metricas_a = validar_resultado(df_a, "A")
metricas_a.update(nome="A", linhas_jdbc=int(linhas_a_jdbc), tempo_total_seg=time.perf_counter() - inicio_a, qt_consultas=1)
df_a.createOrReplaceTempView("vw_publico_possivel_a")

inicio_b = time.perf_counter()
dfs_b = []
linhas_b_jdbc = 0
tempos_faixas_b = {}
for ptc_min, ptc_max, rotulo in NR_PTC_RANGES:
    sql_b = f"""
SELECT CD_CLI, MAX(TIMESTAMP(TS_ATL_TRAN)) AS TS_ATL_TRAN_REF
FROM {FONTE_BLOCO_1A}
WHERE CD_TIP_PSS = 1
  AND TIMESTAMP(TS_ATL_TRAN) >= TIMESTAMP('{data_ini_sql} 00:00:00')
  AND TIMESTAMP(TS_ATL_TRAN) < TIMESTAMP('{data_fim_sql} 00:00:00')
  AND NR_PTC BETWEEN {ptc_min} AND {ptc_max}
GROUP BY CD_CLI
"""
    validar_sql_bloco_1a(sql_b, com_faixa=True)
    inicio_faixa = time.perf_counter()
    df_faixa = conector_db2.sql(sql_b, fetchsize=fetch_efetivo, query_timeout=timeout_efetivo).select("CD_CLI", "TS_ATL_TRAN_REF").persist(StorageLevel.MEMORY_AND_DISK)
    qt_faixa = df_faixa.count()
    linhas_b_jdbc += int(qt_faixa)
    tempos_faixas_b[rotulo] = time.perf_counter() - inicio_faixa
    dfs_b.append(df_faixa)

df_b_faixas = dfs_b[0]
for df_faixa in dfs_b[1:]:
    df_b_faixas = df_b_faixas.unionByName(df_faixa)
df_b = df_b_faixas.groupBy("CD_CLI").agg(F.max("TS_ATL_TRAN_REF").alias("TS_ATL_TRAN_REF")).persist(StorageLevel.MEMORY_AND_DISK)
df_b.count()
metricas_b = validar_resultado(df_b, "B")
metricas_b.update(nome="B", linhas_jdbc=linhas_b_jdbc, tempo_total_seg=time.perf_counter() - inicio_b, qt_consultas=len(NR_PTC_RANGES), faixas=NR_PTC_RANGES, tempos_faixas_seg=tempos_faixas_b)
df_b.createOrReplaceTempView("vw_publico_possivel_b")

print("[V17][BLOCO_1A] metricas A:", metricas_a)
print("[V17][BLOCO_1A] metricas B:", metricas_b)

## Equivalencia A/B

A diferenca e calculada nas duas colunas da feature. Timestamps diferentes para os mesmos clientes tornam o benchmark nao equivalente.

In [ ]:
%%spark

colunas_feature = ["CD_CLI", "TS_ATL_TRAN_REF"]
a_menos_b = df_a.select(*colunas_feature).exceptAll(df_b.select(*colunas_feature)).count()
b_menos_a = df_b.select(*colunas_feature).exceptAll(df_a.select(*colunas_feature)).count()
metricas_equivalencia = {"A_MENOS_B": int(a_menos_b), "B_MENOS_A": int(b_menos_a)}
if a_menos_b != 0 or b_menos_a != 0:
    raise AssertionError(f"BENCHMARK NAO EQUIVALENTE: {metricas_equivalencia}")
print("[V17][BLOCO_1A] equivalencia CD_CLI + TS_ATL_TRAN_REF:", metricas_equivalencia)
print("[V17][BLOCO_1A] nenhuma estrategia e escolhida automaticamente; comparar tempo, volume e falhas.")

## VALIDACOES TEMPORARIAS - BLOCO 1A
### REMOVER/REDUZIR APOS HOMOLOGACAO DA V17

A celula de benchmark valida somente: quantidade de linhas, quantidade de clientes, nulos e duplicidade de `CD_CLI`, nulos e menor/maior `TS_ATL_TRAN_REF`, tempos A/B, linhas JDBC A/B e equivalencia bidirecional. Nenhum atributo de fase futura e calculado.

## Resultado da fase

Resultado temporario: `CD_CLI` + `TS_ATL_TRAN_REF`.

Proxima fase: BLOQUEADA. Apresentar ao lider tecnico antes de criar qualquer outro atributo.

In [ ]:
%%spark

if "df_a" not in globals() or "df_b" not in globals():
    raise RuntimeError("V17 - BLOCO 1A nao finalizado: benchmark A/B nao executado.")

print("============================================================")
print("V17 - BLOCO 1A FINALIZADO")
print("Resultado: CD_CLI + TS_ATL_TRAN_REF")
print("Proxima fase: BLOQUEADA")
print("Aguardando analise do lider tecnico.")
print("STOP")
print("============================================================")